<a href="https://colab.research.google.com/github/IvanMorsin/Forecasting-electrical-power-in-multi-storey-residential-buildings/blob/main/notebook_11_N_beats%2C_N_HiTS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install neuralforecast -q
!pip install kaleido==0.2.1 -q
!pip install workalendar -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.0/287.0 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.2/348.2 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 MB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.2/447.2 kB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 58.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires tornado==6.5.1, but you have tornado 6.5.5 which is incompatible.
   ━━━━━━━

In [ ]:
import pandas as pd
import numpy as np
import torch
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from neuralforecast import NeuralForecast
from neuralforecast.models import NBEATSx, TFT
from neuralforecast.losses.pytorch import MAE
from workalendar.europe import Russia
from tqdm import tqdm
from neuralforecast.models import TFT
import os

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
def save_predictions(ts, house_ids, y_true, y_pred, horizon_name, model_name, filename):
    df_pred = pd.DataFrame({
        'timestamp': ts,
        'house_id':  house_ids,
        'y_true':    y_true,
        'y_pred':    y_pred,
        'horizon':   horizon_name,
        'model':     model_name,
    })
    if os.path.exists(filename):
        df_existing = pd.read_csv(filename)
        df_pred = pd.concat([df_existing, df_pred], ignore_index=True)
    df_pred.to_csv(filename, index=False)

In [ ]:
HOUSE_META = {
    'house_1': {'n_flats': 383, 'n_floors': 12},
    'house_2': {'n_flats': 191, 'n_floors': 12},
    'house_3': {'n_flats': 124, 'n_floors': 12},
    'house_4': {'n_flats': 263, 'n_floors': 12},
    'house_5': {'n_flats': 127, 'n_floors':  7},
    'house_6': {'n_flats': 497, 'n_floors': 25},
    'house_7': {'n_flats': 471, 'n_floors': 17},
    'house_8': {'n_flats': 171, 'n_floors': 23},
}

In [ ]:
df = pd.read_csv("df_final+whether.csv", parse_dates=["timestamp"])
houses = [col for col in df.columns if col.startswith("house_")]

cal = Russia()
def is_holiday(dt):
    if dt.weekday() >= 5:
        return 0
    return int(not cal.is_working_day(dt.date()))

df["is_holiday"] = df["timestamp"].apply(is_holiday)

def evaluate(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    return {"MAE": round(mae, 3), "RMSE": round(rmse, 3), "MAPE": round(mape, 3)}

horizons = {
    "4h":  8, "8h": 16, "24h": 48,
    "7d":  336, "14d": 672, "1m": 1488,
}

print(f"Строк: {len(df)}")
print(f"Дома: {houses}")

Строк: 36260
Дома: ['house_1', 'house_2', 'house_3', 'house_4', 'house_5', 'house_6', 'house_7', 'house_8']


In [ ]:
# neuralforecast требует формат: unique_id | ds | y | exog_features
n = len(df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

dfs = []
for house in houses:
    tmp = pd.DataFrame({
        "unique_id": house,
        "ds": df["timestamp"],
        "y": df[house],
        "temp_c": df["temp_c"],
        "humidity": df["humidity"],
        "cloudiness": df["cloudiness"],
        "hour": df["timestamp"].dt.hour,
        "weekday": df["timestamp"].dt.weekday,
        "month": df["timestamp"].dt.month,
        "is_weekend": (df["timestamp"].dt.weekday >= 5).astype(int),
        "is_holiday": df["is_holiday"],
    })
    dfs.append(tmp)

df_nf = pd.concat(dfs, ignore_index=True)
df_nf['y'] = df_nf.groupby('unique_id')['y'].transform(
    lambda x: x.interpolate(method='linear').ffill().bfill()
)
static_df = pd.DataFrame([
    {
        'unique_id': house,
        'n_flats':   HOUSE_META[house]['n_flats'],
        'n_floors':  HOUSE_META[house]['n_floors'],
    }
    for house in houses
])

# разбивка
df_train = df_nf[df_nf["ds"] < df["timestamp"].iloc[train_end]]
df_val = df_nf[
    (df_nf["ds"] >= df["timestamp"].iloc[train_end]) &
    (df_nf["ds"] < df["timestamp"].iloc[val_end])
]
df_test = df_nf[df_nf["ds"] >= df["timestamp"].iloc[val_end]]
df_test

,unique_id,ds,y,temp_c,humidity,cloudiness,hour,weekday,month,is_weekend,is_holiday
30821,house_1,2019-08-15 03:00:00,31.23,17.000000,81.000000,100.000000,3,3,8,0,0
30822,house_1,2019-08-15 03:30:00,29.61,17.050000,81.833333,99.166667,3,3,8,0,0
30823,house_1,2019-08-15 04:00:00,30.60,17.100000,82.666667,98.333333,4,3,8,0,0
30824,house_1,2019-08-15 04:30:00,28.35,17.150000,83.500000,97.500000,4,3,8,0,0
30825,house_1,2019-08-15 05:00:00,29.37,17.200000,84.333333,96.666667,5,3,8,0,0
...,...,...,...,...,...,...,...,...,...,...,...
290075,house_8,2019-12-06 08:00:00,28.50,2.066667,89.000000,100.000000,8,4,12,0,0
290076,house_8,2019-12-06 08:30:00,28.50,2.083333,89.000000,100.000000,8,4,12,0,0
290077,house_8,2019-12-06 09:00:00,28.50,2.100000,89.000000,100.000000,9,4,12,0,0
290078,house_8,2019-12-06 09:30:00,28.50,2.133333,87.166667,100.000000,9,4,12,0,0


In [ ]:
nbeats_results = {}
cv_dfs = {}

for horizon_name, horizon_points in tqdm(horizons.items(), desc='NBEATSx горизонты'):

    nf = NeuralForecast(
        models=[
            NBEATSx(
                h=horizon_points,
                input_size=336,
                max_steps=500,
                val_check_steps=50,
                early_stop_patience_steps=5,
                scaler_type='minmax',
                accelerator='gpu',
                devices=1,
            )
        ],
        freq='30min'
    )

    df_nf_simple = df_nf[['unique_id', 'ds', 'y']].copy()

    cv_df = nf.cross_validation(
        df=df_nf_simple,
        static_df=static_df,
        val_size=len(df_val) // len(houses),
        test_size=len(df_test) // len(houses),
        n_windows=None,
    )

    cv_dfs[horizon_name] = cv_df.copy()

    for house in houses:
        mask_h = cv_df['unique_id'] == house
        last_cutoff = cv_df.loc[mask_h, 'cutoff'].max()
        mask_test = mask_h & (cv_df['cutoff'] == last_cutoff)

        save_predictions(
            ts=cv_df.loc[mask_test, 'ds'].values,
            house_ids=np.full(mask_test.sum(), house),
            y_true=cv_df.loc[mask_test, 'y'].values,
            y_pred=cv_df.loc[mask_test, 'NBEATSx'].values,
            horizon_name=horizon_name,
            model_name='NBEATSx',
            filename='predictions_nbeats.csv'
        )

    nbeats_results[horizon_name] = {
        house: evaluate(
            cv_df[cv_df['unique_id'] == house]['y'].values,
            cv_df[cv_df['unique_id'] == house]['NBEATSx'].values
        )
        for house in houses
    }

# сводная таблица
rows = []
for horizon_name in horizons.keys():
    for house in houses:
        metrics = nbeats_results[horizon_name][house]
        rows.append({
            'модель': 'NBEATSx',
            'дом': house,
            'горизонт': horizon_name,
            'MAE': metrics['MAE'],
            'RMSE': metrics['RMSE'],
            'MAPE': metrics['MAPE'],
        })

df_nbeats = pd.DataFrame(rows)

for metric in ['MAE', 'RMSE', 'MAPE']:
    pivot = df_nbeats.pivot(index='горизонт', columns='дом', values=metric)
    pivot = pivot.reindex(list(horizons.keys()))
    pivot['Среднее'] = pivot.mean(axis=1).round(2)
    print(f'\nNBEATSx - {metric}:')
    print(pivot.to_string())

df_nbeats.to_csv('results_nbeats.csv', index=False)

NBEATSx горизонты:   0%|          | 0/6 [00:00<?, ?it/s]INFO:lightning_fabric.utilities.seed:Seed set to 1
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:You are using a CUDA device ('NVIDIA A100-SXM4-40GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RA

┏━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name         ┃ Type          ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ loss         │ MAE           │      0 │ train │     0 │
│ 1 │ padder_train │ ConstantPad1d │      0 │ train │     0 │
│ 2 │ scaler       │ TemporalNorm  │      0 │ train │     0 │
│ 3 │ blocks       │ ModuleList    │  3.1 M │ train │     0 │
└───┴──────────────┴───────────────┴────────┴───────┴───────┘

Trainable params: 3.1 M                                                                                            
Non-trainable params: 5.8 K                                                                                        
Total params: 3.1 M                                                                                                
Total estimated model params size (MB): 12                                                                         
Modules in train mode: 31                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

...

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=500` reached.


INFO:pytorch_lightning.utilities.rank_zero:Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.rich_model_summary.RichModelSummary'>]. Skipping setting a default `ModelSummary` callback.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

NBEATSx горизонты: 100%|██████████| 6/6 [05:14<00:00, 52.50s/it]


NBEATSx - MAE:
дом       house_1  house_2  house_3  house_4  house_5  house_6  house_7  house_8  Среднее
горизонт                                                                                 
4h          2.451    1.428    1.085    1.762    1.493    3.419    3.181    1.630     2.06
8h          2.640    1.496    1.125    1.863    1.572    3.650    3.437    1.729     2.19
24h         2.875    1.583    1.177    1.996    1.648    3.970    3.787    1.871     2.36
7d          3.947    1.929    1.386    2.570    2.076    5.169    4.927    2.498     3.06
14d         4.737    2.208    1.548    3.039    2.479    6.190    5.867    2.934     3.63
1m          6.097    2.670    1.813    3.793    3.107    7.937    7.395    3.740     4.57

NBEATSx - RMSE:
дом       house_1  house_2  house_3  house_4  house_5  house_6  house_7  house_8  Среднее
горизонт                                                                                 
4h          3.339    1.923    1.445    2.375    2.030    4.550    4

In [ ]:
import gc
torch.cuda.empty_cache()
gc.collect()

5731

In [ ]:
house = 'house_1'

fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=[f'Горизонт {h}' for h in horizons.keys()],
    vertical_spacing=0.1,
    horizontal_spacing=0.08
)

for i, (hn, hp) in enumerate(horizons.items()):
    row = i // 2 + 1
    col = i % 2 + 1

    house_cv = cv_dfs[hn][cv_dfs[hn]['unique_id'] == 'house_1'].reset_index(drop=True)
    cutoffs = house_cv['cutoff'].unique()

    # берём один cutoff первые hp точек
    subset = house_cv[house_cv['cutoff'] == cutoffs[0]].sort_values('ds')

    fig.add_trace(go.Scatter(
        x=subset['ds'], y=subset['y'],
        mode='lines', name='факт',
        line=dict(color='#1f77b4', width=1.2, shape='hv'),
        showlegend=(i == 0)
    ), row=row, col=col)

    fig.add_trace(go.Scatter(
        x=subset['ds'], y=subset['NBEATSx'],
        mode='lines', name='прогноз NBEATSx',
        line=dict(color='#d62728', width=1.2, shape='hv'),
        showlegend=(i == 0)
    ), row=row, col=col)

    fig.update_xaxes(title_text='Дата', row=row, col=col, title_font=dict(size=9))
    fig.update_yaxes(title_text='кВт',  row=row, col=col, title_font=dict(size=9))

fig.update_layout(
    title='NBEATSx: прогнозные и фактические значения по всем горизонтам (дом 1)',
    width=700, height=900,
    font=dict(size=10),
    margin=dict(l=50, r=20, t=40, b=40),
    legend=dict(font=dict(size=9))
)
fig.show()
fig.write_image('34_nbeats_forecast_all_horizons.png', scale=2)

In [ ]:
from neuralforecast.models import NHITS

In [ ]:
#N-HITS
nhits_results = {}
cv_dfs_nhits  = {}

for horizon_name, horizon_points in tqdm(horizons.items(), desc='N-HiTS горизонты'):
    print(f'\nГоризонт {horizon_name}')

    torch.cuda.empty_cache()
    gc.collect()

    nf = NeuralForecast(
        models=[
            NHITS(
                h=horizon_points,
                input_size=336,
                max_steps=500,
                val_check_steps=50,
                early_stop_patience_steps=5,
                scaler_type='minmax',
                accelerator='gpu',
                devices=1,
            )
        ],
        freq='30min'
    )

    df_nf_simple = df_nf[['unique_id', 'ds', 'y']].copy()

    cv_df = nf.cross_validation(
        df=df_nf_simple,
        static_df=static_df,
        val_size=len(df_val)  // len(houses),
        test_size=len(df_test) // len(houses),
        n_windows=None,
    )

    cv_dfs_nhits[horizon_name] = cv_df.copy()

    # сохраняем только последний cutoff
    for house in houses:
        mask_h      = cv_df['unique_id'] == house
        last_cutoff = cv_df.loc[mask_h, 'cutoff'].max()
        mask_test   = mask_h & (cv_df['cutoff'] == last_cutoff)

        save_predictions(
            ts=cv_df.loc[mask_test, 'ds'].values,
            house_ids=np.full(mask_test.sum(), house),
            y_true=cv_df.loc[mask_test, 'y'].values,
            y_pred=cv_df.loc[mask_test, 'NHITS'].values,
            horizon_name=horizon_name,
            model_name='NHITS',
            filename='predictions_nhits.csv'
        )

    nhits_results[horizon_name] = {
        house: evaluate(
            cv_df[cv_df['unique_id'] == house]['y'].values,
            cv_df[cv_df['unique_id'] == house]['NHITS'].values
        )
        for house in houses
    }

# сводная таблица
rows = []
for horizon_name in horizons.keys():
    for house in houses:
        metrics = nhits_results[horizon_name][house]
        rows.append({
            'модель': 'NHITS',
            'дом': house,
            'горизонт': horizon_name,
            'MAE': metrics['MAE'],
            'RMSE': metrics['RMSE'],
            'MAPE': metrics['MAPE'],
        })

df_NHITS = pd.DataFrame(rows)

for metric in ['MAE', 'RMSE', 'MAPE']:
    pivot = df_NHITS.pivot(index='горизонт', columns='дом', values=metric)
    pivot = pivot.reindex(list(horizons.keys()))
    pivot['Среднее'] = pivot.mean(axis=1).round(2)
    print(f'\nN-HiTS - {metric}:')
    print(pivot.round(3).to_string())

df_NHITS.to_csv('results_NHITS.csv', index=False)

N-HiTS горизонты:   0%|          | 0/6 [00:00<?, ?it/s]


Горизонт 4h


INFO:lightning_fabric.utilities.seed:Seed set to 1
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name         ┃ Type          ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ loss         │ MAE           │      0 │ train │     0 │
│ 1 │ padder_train │ ConstantPad1d │      0 │ train │     0 │
│ 2 │ scaler       │ TemporalNorm  │      0 │ train │     0 │
│ 3 │ blocks       │ ModuleList    │  3.2 M │ train │     0 │
└───┴──────────────┴───────────────┴────────┴───────┴───────┘

Trainable params: 3.2 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 3.2 M                                                                                                
Total estimated model params size (MB): 12                                                                         
Modules in train mode: 34                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

...

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=500` reached.


INFO:pytorch_lightning.utilities.rank_zero:Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.rich_model_summary.RichModelSummary'>]. Skipping setting a default `ModelSummary` callback.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

N-HiTS горизонты: 100%|██████████| 6/6 [05:16<00:00, 52.80s/it]


N-HiTS - MAE:
дом       house_1  house_2  house_3  house_4  house_5  house_6  house_7  house_8  Среднее
горизонт                                                                                 
4h          2.449    1.432    1.094    1.770    1.496    3.381    3.238    1.640     2.06
8h          2.672    1.508    1.132    1.892    1.583    3.693    3.503    1.743     2.22
24h         2.857    1.573    1.171    1.986    1.640    3.940    3.794    1.860     2.35
7d          3.926    1.917    1.377    2.563    2.061    5.162    4.945    2.496     3.06
14d         4.525    2.113    1.488    2.891    2.372    5.855    5.493    2.854     3.45
1m          5.629    2.519    1.719    3.484    2.959    7.484    6.708    3.587     4.26

N-HiTS - RMSE:
дом       house_1  house_2  house_3  house_4  house_5  house_6  house_7  house_8  Среднее
горизонт                                                                                 
4h          3.317    1.911    1.448    2.366    2.030    4.515    4.2

In [ ]:
# график N-HiTS
torch.cuda.empty_cache()
gc.collect()

fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=[f'Горизонт {h}' for h in horizons.keys()],
    vertical_spacing=0.1,
    horizontal_spacing=0.08
)

for i, (hn, hp) in enumerate(horizons.items()):
    row = i // 2 + 1
    col = i % 2 + 1

    house_cv = cv_dfs_nhits[hn][
        cv_dfs_nhits[hn]['unique_id'] == 'house_1'
    ].reset_index(drop=True)

    cutoffs_sorted = sorted(house_cv['cutoff'].unique())
    mid_cutoff = cutoffs_sorted[len(cutoffs_sorted) // 2]

    subset = house_cv[
        house_cv['cutoff'] == mid_cutoff
    ].sort_values('ds').head(hp)

    fig.add_trace(go.Scatter(
        x=subset['ds'], y=subset['y'],
        mode='lines', name='факт',
        line=dict(color='#1f77b4', width=1.2, shape='hv'),
        showlegend=(i == 0)
    ), row=row, col=col)

    fig.add_trace(go.Scatter(
        x=subset['ds'], y=subset['NHITS'],
        mode='lines', name='прогноз N-HiTS',
        line=dict(color='#d62728', width=1.2, shape='hv'),
        showlegend=(i == 0)
    ), row=row, col=col)

    fig.update_xaxes(title_text='Дата', row=row, col=col, title_font=dict(size=9))
    fig.update_yaxes(title_text='кВт',  row=row, col=col, title_font=dict(size=9))

fig.update_layout(
    title='N-HiTS: прогнозные и фактические значения по всем горизонтам (дом 1)',
    width=700, height=900,
    font=dict(size=10),
    margin=dict(l=50, r=20, t=40, b=40),
    legend=dict(font=dict(size=9))
)
fig.show()
fig.write_image('35_nhits_forecast_all_horizons.png', scale=2)